# Занятие 8. Точки интереса: детекторы

Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026
Лекция-опора: L8 «Точки интереса: детекторы» · 90 минут в классе (последние 20 — клиника ДЗ 2) · ведёт ассистент

---

Сегодня считаем **матрицу вторых моментов** (структурный тензор) и покажем числом, как собственные значения отличают угол от края и ровного места. Потом соберём **детектор Харриса** — `R = det(M) − k·trace²(M)` — и проверим, что он совпадает с `cv2.cornerHarris`. В третьей части реализуем **детектор Ши-Томаси** — `R = min(λ₁, λ₂)` — и сравним с `cv2.goodFeaturesToTrack`. В четвёртой — проверим **инвариантность к повороту**: повернём кадр, найдём углы снова и измерим повторяемость.

Заготовка начинается на слайдах 24 и 34 лекции («Матрица вторых моментов» и «Harris Corner Detector»).

| Минуты | Что происходит |
|--------|----------------|
| 0–12 | Ассистент разбирает опорный пример: собственные значения и три типа точек |
| 12–58 | Вы делаете TODO 1–4 |
| 58–67 | Разбор решения |
| 67–87 | Клиника: ДЗ 2 «Сканер документов» — прогресс и связь с лекциями 7–8 |
| 87–90 | Зачёт |

Ничего сдавать не нужно: зачёт ставится в классе по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется в репозитории сразу после занятия.
**Занятие 9 — дескрипторы и сопоставление.**

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt (docs/setup-guide.md)")

import os, time, timeit
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём данные

Фото здания ИУ `lectures/L08-features/demo/iu-building.jpg` (1920 × 1279) читается в сером;
кроп лестницы 256 × 256 тот же, что в занятиях 5–7. Для проверки детекторов на известных углах
порождается синтетическое изображение с чёткими углами, краями и ровными областями.

Источники по порядку: репозиторий рядом → скачать с GitHub (Colab) → синтетическая сцена.

In [ ]:
LECTURE_DIR = "lectures/L08-features/demo"
CANDIDATE_DIRS = ["../../" + LECTURE_DIR,
                  "../../../iu-intro-cv/" + LECTURE_DIR,
                  "data"]
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
FILE = "iu-building.jpg"
CROP = (slice(880, 1136), slice(1300, 1556))


def synthetic_photo(h=1279, w=1920):
    yy, xx = np.mgrid[:h, :w].astype(np.float32)
    img = 120 + 60 * np.sin(xx / 97.0) * np.cos(yy / 71.0) + 0.03 * xx
    for cx, cy, r in ((500, 400, 220), (1400, 900, 300), (1000, 250, 120)):
        img[np.hypot(xx - cx, yy - cy) < r] += 50
    img[880:1136:16, 1300:1556] -= 70
    img[880:1136, 1300:1556:32] -= 50
    return np.uint8(np.clip(img, 0, 255))


def load_photo():
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, FILE)
        if os.path.exists(p):
            return cio.imread(p, "gray"), d
    try:
        import urllib.request
        os.makedirs("data", exist_ok=True)
        p = os.path.join("data", FILE)
        urllib.request.urlretrieve(RAW_URL + FILE, p)
        return cio.imread(p, "gray"), RAW_URL
    except Exception as e:
        print("файл лекции не скачался (%s) — синтетическая сцена" % type(e).__name__)
        return synthetic_photo(), "синтетика"


def make_corner_image(size=128):
    """Синтетическое изображение: угол, край, ровная область."""
    img = np.zeros((size, size), np.float64)
    img[:size//2, :size//2] = 200
    img[size//2:, size//4:3*size//4] = 150
    return img


photo, SRC = load_photo()
assert photo.shape == (1279, 1920) and photo.dtype == np.uint8
crop = photo[CROP].astype(np.float64)
corner_img = make_corner_image()
print("источник:", SRC, "· кадр", photo.shape, photo.dtype, "· кроп", crop.shape)
viz.grid({"кроп лестницы": crop / 255, "синтетика: угол + край": corner_img / 255}, cols=2, size=3.4)

## 2. Опорный пример — разбирает ассистент

Здесь всё написано, задача — **понять каждую строку**.

**Матрица вторых моментов** (слайд 24): для каждого пикселя считается матрица `M = [[A, C], [C, B]]`, где
`A = Σ Ix²`, `B = Σ Iy²`, `C = Σ Ix·Iy` (сумма по окну). Собственные значения `λ₁, λ₂` этой матрицы говорят,
что это за точка:

- **оба малы** → ровная область (нет градиента ни в одном направлении)
- **одно большое, другое малое** → край (градиент только поперёк)
- **оба большие** → угол (градиент в двух направлениях)

**Детектор Харриса** (слайд 34): вместо явного вычисления собственных значений использует функцию отклика:
`R = det(M) − k·trace²(M) = (A·B − C²) − k·(A + B)²`, где `k ≈ 0.04–0.06`.

In [ ]:
img_test = corner_img
Ix = cv2.Sobel(img_test, cv2.CV_64F, 1, 0, ksize=3)
Iy = cv2.Sobel(img_test, cv2.CV_64F, 0, 1, ksize=3)

A = cv2.boxFilter(Ix * Ix, -1, (7, 7))
B = cv2.boxFilter(Iy * Iy, -1, (7, 7))
C = cv2.boxFilter(Ix * Iy, -1, (7, 7))

trace = A + B
disc = np.sqrt(((A - B) / 2) ** 2 + C ** 2)
lam1 = trace / 2 + disc
lam2 = trace / 2 - disc

pts = {"угол (64,64)": (64, 64), "край (96,64)": (96, 64), "ровное (32,96)": (32, 96)}
print("%-16s %10s %10s %12s" % ("точка", "λ₁", "λ₂", "тип"))
for name, (y, x) in pts.items():
    l1, l2 = lam1[y, x], lam2[y, x]
    kind = "угол" if l1 > 100 and l2 > 100 else ("край" if max(l1, l2) > 100 else "ровное")
    print("%-16s %10.1f %10.1f %12s" % (name, l1, l2, kind))

viz.grid({"λ₁ (большее)": np.abs(lam1) / (np.abs(lam1).max() + 1e-9),
          "λ₂ (меньшее)": np.abs(lam2) / (np.abs(lam2).max() + 1e-9)}, cols=2, size=3.4)

### Вопрос, на который отвечаем вслух

Почему для определения угла нужны **оба** собственных значения, а не просто большая магнитуда градиента?
(Ответ — слайд 22: край тоже даёт большой градиент, но только в одном направлении; угол — в двух.)

---

## TODO 1 — матрица вторых моментов (≈ 10 минут)

Слайд 24. **Структурный тензор** — матрица `M = [[A, C], [C, B]]` для каждого пикселя, где компоненты усреднены по окну.

Напишите `moment_matrix(img, ksize=3, block_size=7)` → кортеж `(A, B, C)`:

Скелет — четыре шага:

1. привести кадр к `float64`;
2. `cv2.Sobel` по `x` и по `y` с `ksize`;
3. квадраты и произведение: `Ix²`, `Iy²`, `Ix·Iy`;
4. усреднение по окну `block_size × block_size`: `cv2.boxFilter(..., (block_size, block_size))`.

> **Подсказка.** `cv2.boxFilter(src, ddepth=-1, ksize=(block_size, block_size), normalize=True)` — среднее по окну.

In [ ]:
def moment_matrix(img, ksize=3, block_size=7):
    """Структурный тензор: (A, B, C) где M = [[A, C], [C, B]], усреднение по окну."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 6–8 строк: float64; Sobel x и y; квадраты и произведение; boxFilter по окну; return (A, B, C)
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────


A, B, C = moment_matrix(corner_img)
assert A.shape == corner_img.shape and B.shape == corner_img.shape and C.shape == corner_img.shape, "формы должны совпадать с входом"
assert A.min() >= 0 and B.min() >= 0, "Ix² и Iy² неотрицательны, их среднее тоже"
flat_region = (slice(100, 120), slice(10, 30))
assert A[flat_region].max() < 1.0 and B[flat_region].max() < 1.0, "на ровном участке градиент мал"
corner_val = A[64, 64] + B[64, 64]
edge_val = A[96, 64] + B[96, 64]
flat_val = A[110, 20] + B[110, 20]
assert corner_val > edge_val * 0.5, "на углу сумма собственных значений больше, чем на крае"
assert edge_val > flat_val * 10, "на крае градиент значительно больше, чем на ровном участке"
print("TODO 1 ✔ · угол: A+B=%.1f · край: A+B=%.1f · ровное: A+B=%.1f" % (corner_val, edge_val, flat_val))

---

## TODO 2 — детектор Харриса (≈ 12 минут)

Слайд 34. **Функция отклика Харриса:** `R = det(M) − k·trace²(M) = (A·B − C²) − k·(A + B)²`.

Напишите `harris_response(img, ksize=3, block_size=7, k=0.04)` → карта отклика `R` (float64).

Скелет — три шага:

1. вызвать `moment_matrix(img, ksize, block_size)` → `(A, B, C)`;
2. `det = A·B − C²`, `trace = A + B`;
3. `R = det − k · trace²`.

> **Подсказка.** `det(M) = A·B − C²`, `trace(M) = A + B`. Параметр `k` обычно 0.04–0.06.

In [ ]:
def harris_response(img, ksize=3, block_size=7, k=0.04):
    """Функция отклика Харриса: R = det(M) − k·trace²(M)."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 4–5 строк: moment_matrix; det; trace; R = det − k·trace²; return R
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


R = harris_response(corner_img)
assert R.shape == corner_img.shape and R.dtype == np.float64, "карта отклика — того же размера, float64"
R_corner = R[64, 64]
R_flat = R[110, 20]
R_edge = R[96, 64]
assert R_corner > 0, "на углу R > 0 (corner)"
assert R_flat <= R_corner, "на ровном участке R не больше, чем на углу"
crop_f32 = np.float32(crop)
R_mine = harris_response(crop, ksize=3, block_size=3, k=0.04)
R_cv = cv2.cornerHarris(crop_f32, blockSize=3, ksize=3, k=0.04)
corr = np.corrcoef(R_mine.ravel(), R_cv.ravel())[0, 1]
assert corr > 0.9, "корреляция с cv2.cornerHarris должна быть > 0.9 (сейчас %.3f)" % corr
print("TODO 2 ✔ · R(угол)=%.1f · R(край)=%.1f · R(ровное)=%.1f · корреляция с cv2: %.3f" % (R_corner, R_edge, R_flat, corr))
viz.grid({"R Харриса (наш)": np.abs(R) / (np.abs(R).max() + 1e-9),
          "R Харриса (кроп)": np.abs(R_mine) / (np.abs(R_mine).max() + 1e-9)}, cols=2, size=3.4)

### Углы на реальном кадре: порог + NMS

После карты отклика — порог и подавление не-максимумов. `cv2.cornerHarris` возвращает карту,
углы — локальные максимумы выше порога.

In [ ]:
crop_f32 = np.float32(crop)
dst = cv2.cornerHarris(crop_f32, blockSize=3, ksize=3, k=0.04)
threshold = 0.01 * dst.max()
corners_mask = dst > threshold
print("пикселей-углов (> 1%% от максимума): %d" % corners_mask.sum())
vis = cv2.cvtColor(np.uint8(np.clip(crop, 0, 255)), cv2.COLOR_GRAY2BGR)
vis[corners_mask] = [0, 0, 255]
viz.grid({"углы Харриса на кропе": vis}, cols=1, size=5)

---

## TODO 3 — детектор Ши-Томаси (≈ 10 минут)

Слайд 45. **Функция отклика Ши-Томаси:** `R = min(λ₁, λ₂)`, где `λ₁, λ₂` — собственные значения матрицы `M`.

Напишите `shi_tomasi_response(img, ksize=3, block_size=7)` → карта отклика `R` (float64).

Скелет — три шага:

1. вызвать `moment_matrix(img, ksize, block_size)` → `(A, B, C)`;
2. собственные значения: `λ = (A+B)/2 ± sqrt(((A−B)/2)² + C²)`;
3. `R = min(λ₁, λ₂)` — меньшее собственное значение.

> **Подсказка.** `λ₁ = (A+B)/2 + sqrt(((A−B)/2)² + C²)`, `λ₂ = (A+B)/2 − sqrt(...)`. `R = λ₂`.

In [ ]:
def shi_tomasi_response(img, ksize=3, block_size=7):
    """Функция отклика Ши-Томаси: R = min(λ₁, λ₂)."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 5–6 строк: moment_matrix; собственные значения через формулу; return min(λ₁, λ₂)
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


R_st = shi_tomasi_response(corner_img)
assert R_st.shape == corner_img.shape, "карта отклика — того же размера, что вход"
R_st_corner = R_st[64, 64]
R_st_edge = R_st[96, 64]
R_st_flat = R_st[110, 20]
assert R_st_corner > R_st_edge, "на углу min(λ₁,λ₂) больше, чем на крае"
assert R_st_edge > R_st_flat, "на крае min(λ₁,λ₂) больше, чем на ровном участке"

corners_cv = cv2.goodFeaturesToTrack(np.float32(crop), maxCorners=50, qualityLevel=0.01, minDistance=10)
print("TODO 3 ✔ · R_st(угол)=%.1f · R_st(край)=%.1f · R_st(ровное)=%.1f · cv2.goodFeaturesToTrack: %d углов" % (R_st_corner, R_st_edge, R_st_flat, len(corners_cv) if corners_cv is not None else 0))
vis = cv2.cvtColor(np.uint8(np.clip(crop, 0, 255)), cv2.COLOR_GRAY2BGR)
if corners_cv is not None:
    for c in corners_cv[:50]:
        x, y = c.ravel().astype(int)
        cv2.circle(vis, (x, y), 4, (0, 255, 0), -1)
viz.grid({"goodFeaturesToTrack (Ши-Томаси)": vis}, cols=1, size=5)

### Вопрос, на который отвечаем вслух

Почему Ши-Томаси использует `min(λ₁, λ₂)`, а не `λ₁ + λ₂` или `λ₁ · λ₂`?
(Ответ — слайд 45: `min` гарантирует, что **оба** собственных значения большие; сумма или произведение не защищают от края.)

---

## TODO 4 — инвариантность к повороту (≈ 10 минут)

Слайды 37–38. Детектор Харриса **инвариантен к повороту**, но **не инвариантен к масштабу** (слайд 39).

Напишите `repeatability(img, angle, block_size=7, ksize=3, k=0.04, threshold_frac=0.01)` → доля углов, найденных повторно после поворота.

Скелет — четыре шага:

1. найти углы Харриса на исходном кадре: `harris_response` → порог → координаты;
2. повернуть кадр на `angle` градусов: `cv2.warpAffine` с матрицей поворота;
3. найти углы Харриса на повёрнутом кадре;
4. для каждого угла исходного кадра: повернуть его координаты, найти ближайший угол в повёрнутом кадре; если расстояние < 5 px — засчитать. Вернуть долю.

> **Подсказка.** Матрица поворота: `cv2.getRotationMatrix2D(center, angle, 1.0)`. Повернуть координаты точек: `M @ [x, y, 1]`.

In [ ]:
def harris_points(img, block_size=7, ksize=3, k=0.04, threshold_frac=0.01):
    """Координаты углов Харриса: (N, 2) массив [y, x]."""
    R = harris_response(img, ksize, block_size, k)
    mask = R > threshold_frac * R.max()
    ys, xs = np.where(mask)
    return np.column_stack([ys, xs])


def repeatability(img, angle, block_size=7, ksize=3, k=0.04, threshold_frac=0.01):
    """Повторяемость углов Харриса после поворота на angle градусов."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10–14 строк: углы на исходном; поворот; углы на повёрнутом; поворот координат; поиск ближайших; доля < 5 px
    raise NotImplementedError("TODO 4")
    # ───────────────────────────────────────────────────────────


rep0 = repeatability(crop, angle=0)
assert rep0 > 0.9, "при повороте на 0° повторяемость должна быть ≈ 1.0 (сейчас %.2f)" % rep0
rep30 = repeatability(crop, angle=30)
assert rep30 > 0.3, "при повороте на 30° повторяемость Харриса должна быть заметной (сейчас %.2f)" % rep30
print("TODO 4 ✔ · повторяемость: 0° → %.2f · 30° → %.2f (Харрис инвариантен к повороту, слайд 37)" % (rep0, rep30))

### Масштаб: Харрис НЕ инвариантен (слайд 39)

При увеличении кадра в 2 раза повторяемость падает до ~20% (слайд 41). Это фундаментальное ограничение детектора Харриса, которое решают масштабно-инвариантные детекторы (SIFT, SURF — L9).

In [ ]:
h, w = crop.shape[:2]
small = cv2.resize(crop, (w // 2, h // 2), interpolation=cv2.INTER_AREA)
pts_big = harris_points(crop, block_size=7, ksize=3, k=0.04, threshold_frac=0.01)
pts_small = harris_points(small, block_size=7, ksize=3, k=0.04, threshold_frac=0.01)
if len(pts_big) > 0 and len(pts_small) > 0:
    pts_big_scaled = pts_big / 2.0
    dists = np.sqrt(((pts_big_scaled[:, None, :] - pts_small[None, :, :]) ** 2).sum(2))
    rep_scale = float((dists.min(axis=1) < 5).mean())
else:
    rep_scale = 0.0
print("повторяемость при уменьшении ×2: %.2f (по слайду 41: ~0.20)" % rep_scale)
print("Харрис НЕ инвариантен к масштабу — это решают детекторы с выбором масштаба (SIFT, L9)")

### Вопрос, на который отвечаем вслух

Детектор Харриса инвариантен к аффинным изменениям яркости (слайд 38), но не к масштабу (слайд 39).
Почему изменение яркости не влияет на собственные значения, а изменение масштаба — влияет?
(Ответ: собственные значения пропорциональны квадрату градиента; при умножении кадра на константу `c` градиенты умножаются на `c`, собственные значения — на `c²`, но положение локальных максимумов `R` не меняется. При изменении масштаба окно «видит» другую структуру: угол превращается в кривую.)

---

## Клиника: ДЗ 2 «Сканер документов» — прогресс и связь с лекциями 7–8 (последние 20 минут)

**ДЗ 2 «Сканер документов»** выдано на занятии 7 (вт 13.10). Что из занятий 7–8 туда идёт:

| Тема | Где в ДЗ 2 | Правило |
|------|------------|---------|
| `cv2.Canny` + `findContours` | поиск границ документа | слайд 31 из L7: пороги из гистограммы градиента |
| `cv2.HoughLinesP` | поиск прямых сторон | слайд 59 из L7: `minLineLength`, `maxLineGap` |
| `cv2.approxPolyDP` | поиск 4 вершин контура | слайд 32 из L7: `epsilon = 0.02 * arcLength` |
| Углы документа | выравнивание перспективы | `cv2.getPerspectiveTransform` (L4); углы = точки интереса |
| Детекторы Харриса/Ши-Томаси | альтернатива контурам для углов | сегодня: если контур не найден, углы Харриса дают опорные точки |

**Чек-лист перед сдачей ДЗ 2:**

- `pytest tests -q` — тесты на синтетике;
- на кадрах курса: `cv2.Canny` находит контур документа, `approxPolyDP` возвращает 4 вершины;
- `cv2.getPerspectiveTransform` + `cv2.warpPerspective` выравнивают документ.

**Связь с лекцией 8:** детекторы углов — это то, что будет нужно в ДЗ 3 и дальше (панорамы, сопоставление):
углы документа, найденные `approxPolyDP`, — это и есть «точки интереса» в прикладном смысле.
Детектор Харриса найдёт их автоматически, если контур не замкнут.

### `cornerSubPix`: уточнение до субпиксельной точности (слайд 44)

In [ ]:
corners = cv2.goodFeaturesToTrack(np.float32(crop), maxCorners=20, qualityLevel=0.01, minDistance=15)
if corners is not None:
    corners_refined = cv2.cornerSubPix(
        np.float32(crop), corners.astype(np.float32),
        winSize=(5, 5), zeroZone=(-1, -1),
        criteria=(cv2.TERM_CRITERIA_MAX_ITER | cv2.TERM_CRITERIA_EPS, 30, 0.001))
    vis = cv2.cvtColor(np.uint8(np.clip(crop, 0, 255)), cv2.COLOR_GRAY2BGR)
    for c in corners:
        x, y = c.ravel().astype(int)
        cv2.circle(vis, (x, y), 4, (0, 0, 255), -1)
    for c in corners_refined:
        x, y = c.ravel()
        cv2.circle(vis, (int(round(x)), int(round(y))), 2, (0, 255, 0), -1)
    shift = np.abs(corners_refined.reshape(-1, 2) - corners.reshape(-1, 2)).max()
    print("максимальный сдвиг при субпиксельном уточнении: %.2f px" % shift)
    viz.grid({"углы: красные — целочисленные, зелёные — субпиксель": vis}, cols=1, size=5)

### Вопросы клиники

Принесли вопрос по ДЗ 2 — задавайте сейчас. Если вопросов нет, разбираем чек-лист выше и связь сегодняшней лекции с ДЗ 2. С занятия 5 ассистент принимает **показ ДЗ**: одно ДЗ за семестр, две минуты — запуск команды и объяснение одного места кода.

---

## ⭐ Звёздочка (по желанию, +0.5, до конца следующей недели)

**Субпиксельное уточнение углов** (слайд 44). `cv2.cornerSubPix` итеративно уточняет положение угла:
на каждой итерации ищет направление, в котором сумма `(∇I · (p − q))²` минимальна по всем пикселям окна.

Напишите `corner_subpix_manual(img, corners, win_size=5, max_iter=30, eps=0.001)` —
возвращает уточнённые координаты углов. Ячейка ниже сверит результат с `cv2.cornerSubPix`.

Сдаётся ноутбук с функцией и одной фразой: почему метод сходится (минимизация по линейной модели яркости).

In [ ]:
def corner_subpix_manual(img, corners, win_size=5, max_iter=30, eps=0.001):
    """Субпиксельное уточнение углов: итеративная минимизация."""
    raise NotImplementedError("⭐ звёздочка: субпиксельное уточнение углов")


try:
    corners_test = cv2.goodFeaturesToTrack(np.float32(crop), maxCorners=10, qualityLevel=0.01, minDistance=20)
    refined_mine = corner_subpix_manual(np.float32(crop), corners_test)
except NotImplementedError:
    print("⭐ не заполнена — это необязательная часть; замеры появятся, когда функция будет написана")
else:
    refined_cv = cv2.cornerSubPix(np.float32(crop), corners_test.astype(np.float32),
                                  winSize=(5, 5), zeroZone=(-1, -1),
                                  criteria=(cv2.TERM_CRITERIA_MAX_ITER | cv2.TERM_CRITERIA_EPS, 30, 0.001))
    diff = np.abs(refined_mine.reshape(-1, 2) - refined_cv.reshape(-1, 2)).max()
    assert diff < 0.5, "расхождение с cv2.cornerSubPix должно быть < 0.5 px (сейчас %.3f)" % diff
    print("⭐ субпиксельное уточнение: расхождение с cv2.cornerSubPix %.3f px" % diff)

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–4 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, **как собственные значения отличают угол от края** и почему Харрис **инвариантен к повороту, но не к масштабу**;
- [ ] на вопросы «почему `min(λ₁, λ₂)`, а не сумма», «что делает параметр `k` в формуле Харриса» и «почему повторяемость падает при масштабе» есть внятный ответ.

2 % итоговой оценки, ставится в классе. Досылать ничего не нужно. **Занятие 9 — дескрипторы и сопоставление.**